In [33]:
# First of all, install the necessary packages. We will be using Selenium to automate a web browser in order to scrape LinkedIn jobs. 
%pip install selenium pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd 

import time 

from selenium import webdriver 
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait

In [ ]:
def scrape_linkedin(num_jobs, delay, testing):
    
    '''Gathers jobs as a dataframe, scraped from LinkedIn'''
    
    # Initializing the webdriver.
    options = webdriver.ChromeOptions()
    if not testing:
        options.add_argument('--headless')
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()

    # We start at the homepage since the site may force us to go there anyways.
    driver.get("https://www.linkedin.com/?trk=guest_homepage-basic_nav-header-logo")
    jobs = [] # We store our job listings here. It will be a list of dictionaries.

    time.sleep(delay) # The waiting time (in seconds) between requests. Ensure it is high enough to load pages. 

    # Click on "Jobs" button.
    driver.find_elements(By.XPATH, "//icon[@class='top-nav-link__icon flex h-3 w-3 flex-shrink-0 justify-center lazy-loaded']")[3].click()
    
    # Wait until the sign-in popup appears then clear it. If it doesn't appear after some time then continue on. 
    try:
        WebDriverWait(driver, 300).until(lambda d: d.find_element(By.XPATH, "//*[contains(text(), 'Sign in to view more jobs')]").is_displayed()) 
        time.sleep(delay)
        ActionChains(driver).send_keys(Keys.ESCAPE).perform() 
        time.sleep(delay)
    except TimeoutException: 
        pass
    
    # Input the value in the "keyword" variable inside in job search box.
    job_search_box = driver.find_element(By.XPATH, "//input[@aria-label='Search job titles or companies']")
    job_search_box.clear()
    time.sleep(delay)
    job_search_box.send_keys('Data OR Analyst')
    time.sleep(delay)

    # Clear the location search box and input "Australia" then hit enter.
    location_search_box = driver.find_element(By.XPATH, "//input[@aria-label='Location']")
    location_search_box.clear()
    time.sleep(delay)
    location_search_box.send_keys("Australia")
    time.sleep(delay)
    location_search_box.send_keys(Keys.ENTER)
    time.sleep(delay) 

    # Scroll to the bottom of the page until it no longer loads.
    current_height = driver.execute_script("return document.body.scrollHeight") # Get current height of page.
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight)") # Scroll to the bottom to load.
        time.sleep(delay)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == current_height: # Only possible when the page no longer loads, i.e., we have reached the end.
            break
        current_height = new_height

    # Click "See more jobs". Sometimes when clicking the button the page does not load. If this happens above a threshold then we stop clicking.  
    while True:
        driver.find_element(By.XPATH, "//button[@aria-label='See more jobs']").click() # Click the button.
        time.sleep(delay)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == current_height:
            break
        current_height = new_height

    driver.execute_script("window.scrollTo(0, 0)")

    time.sleep(delay)

    # 'job_postings' contains all the jobs on the page.
    job_postings = driver.find_elements(By.XPATH, "//ul[@class='jobs-search__results-list']/*")
    
    # If fewer jobs are found than what is requested, tell the user. Otherwise take the first "num_jobs" from the page.
    if len(job_postings)<num_jobs:
        print("The requested number of jobs is {0} but the search found {1} jobs. The scraper will therefore return only {1} jobs.".format(num_jobs, len(job_postings)))
    else:
        job_postings = job_postings[:num_jobs]

    for job_posting in job_postings: # Going through each job on the page.

        # Wait until the sign-in popup appears then clear it. If it doesn't appear after some time then continue on. 
        if driver.find_element(By.XPATH, "//*[contains(text(), 'Sign in to view more jobs')]").is_displayed():
            time.sleep(delay)
            ActionChains(driver).send_keys(Keys.ESCAPE).perform() 
            time.sleep(delay)

        job_posting.click() # Click the job listing to open its contents. 
        time.sleep(delay)

        # Collect information on the job title, location, and description. These should always be available under each job.
        date = driver.find_element(By.XPATH, "//span[contains(@class, 'posted-time-ago__text')]").text
        title = driver.find_element(By.XPATH, "//h2[@class='top-card-layout__title font-sans text-lg papabear:text-xl font-bold leading-open text-color-text mb-0 topcard__title']").text
        company = driver.find_element(By.XPATH, "//a[@class='topcard__org-name-link topcard__flavor--black-link']").text
        location = driver.find_element(By.XPATH, "//span[@class='topcard__flavor topcard__flavor--bullet']").text
        industry = driver.find_element(By.XPATH, "//li[@class='description__job-criteria-item' and contains(., 'Industries')]").find_element(By.XPATH, './span').text
        employment_type = driver.find_element(By.XPATH, "//li[@class='description__job-criteria-item' and contains(., 'Employment type')]").find_element(By.XPATH, './span').text
        description = driver.find_element(By.XPATH, "//div[@class='description__text description__text--rich']/section/div").text
        try:
            salary = driver.find_element(By.XPATH, "//div[@class='salary compensation__salary']").text
        except NoSuchElementException:
            salary = 'Unavailable'

        # Add the job to 'jobs'.
        jobs.append({'Date': date,
                     'Title': title,
                     'Company Name': company,
                     'Location': location,
                     'Industry': industry,
                     'Employment type': employment_type,
                     'Description': description,
                     'Salary': salary})

        # Printing for debugging.
        if testing:
            print("Progress: {}".format(str(len(jobs)) + "/" + str(len(job_postings))))
            print("Date: {}".format(date))
            print("Title: {}".format(title))
            print("Company: {}".format(company))
            print("Location: {}".format(location))
            print("Industry: {}".format(industry))
            print("Employment type: {}".format(employment_type))
            print("Description: {}".format(description[:50] + '...'))
            print("Salary: {}".format(salary))
            print()
            
    return pd.DataFrame(jobs) 

In [36]:
# Start the scraping. 
df = scrape_linkedin(num_jobs=250, delay = 3, testing=True)
df

Progress: 1/250
Date: 5 days ago
Title: Senior Data Analyst
Company: Cleanaway Waste Management
Location: Melbourne, Victoria, Australia
Industry: Environmental Services
Employment type: Full-time
Description: Cleanaway Waste Management transforms waste into r...
Salary: Unavailable

Progress: 2/250
Date: 2 days ago
Title: Business Intelligence Data Analyst
Company: The NRMA
Location: Sydney Olympic Park, New South Wales, Australia
Industry: Civic and Social Organizations
Employment type: Full-time
Description: What you’ll do

The BI Data Analyst plays a truste...
Salary: Unavailable

Progress: 3/250
Date: 1 month ago
Title: Data Analyst
Company: BINGO INDUSTRIES
Location: Sydney, New South Wales, Australia
Industry: Environmental Services
Employment type: Full-time
Description: BINGO is looking for a hands-on Data Analyst to jo...
Salary: Unavailable

Progress: 4/250
Date: 1 week ago
Title: Senior Data Analyst
Company: Metro Tasmania Pty Ltd
Location: Hobart, Tasmania, Australia
Indus

,Date,Title,Company Name,Location,Industry,Employment type,Description,Salary
0,5 days ago,Senior Data Analyst,Cleanaway Waste Management,"Melbourne, Victoria, Australia",Environmental Services,Full-time,Cleanaway Waste Management transforms waste in...,Unavailable
1,2 days ago,Business Intelligence Data Analyst,The NRMA,"Sydney Olympic Park, New South Wales, Australia",Civic and Social Organizations,Full-time,What you’ll do\n\nThe BI Data Analyst plays a ...,Unavailable
2,1 month ago,Data Analyst,BINGO INDUSTRIES,"Sydney, New South Wales, Australia",Environmental Services,Full-time,BINGO is looking for a hands-on Data Analyst t...,Unavailable
3,1 week ago,Senior Data Analyst,Metro Tasmania Pty Ltd,"Hobart, Tasmania, Australia",Urban Transit Services,Full-time,Join the team at Metro Tasmania – Your Career ...,Unavailable
4,1 week ago,Senior Data Analyst,Go1,"Melbourne, Victoria, Australia",E-Learning Providers,Full-time,"At Go1, we’re on a mission to help organizatio...",Unavailable
...,...,...,...,...,...,...,...,...
245,3 days ago,Business Analyst/Senior Business Analyst,Gunn Agri Partners,"Albury, New South Wales, Australia",Farming,Full-time,Wilga Farming Australia (WFA) is seeking a pro...,Unavailable
246,5 days ago,Business Analyst/Senior Business Analyst,Gunn Agri Partners,"Queens Park, New South Wales, Australia",Farming,Full-time,Wilga Farming Australia (WFA) is seeking a pro...,Unavailable
247,1 week ago,Associate Performance Attribution Analyst,Downer,"Auburn, New South Wales, Australia",Rail Transportation,Full-time,We currently have an exciting Graduate entry-l...,Unavailable
248,1 week ago,Associate Performance Attribution Analyst,Downer,"Auburn, New South Wales, Australia",Rail Transportation,Full-time,We currently have an exciting Graduate entry-l...,Unavailable
